In [2]:
# Importar librerias
import pandas as pd
import plotly.express as px
import joblib
from sklearn.model_selection import train_test_split

# Semilla fija
RANDOM_STATE = 42

# Cargar datos
df = pd.read_csv('data/players.csv')

print(f"Datos listos. Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

Datos listos. Filas: 50149, Columnas: 26


## 0. Executive summary
El FC Barcelona necesita optimizar su presupuesto de fichajes ajustándose al Fair Play financiero. Este modelo de Machine Learning (Regresión) predice el valor de mercado real de los futbolistas basándose en sus características y rendimiento, superando la regla actual de tasación basada en promedios históricos. Esto permite al Director Deportivo detectar talentos infravalorados en ligas secundarias antes de que su precio se dispare. La principal limitación es que el modelo no contempla variables intangibles como la disciplina o el encaje psicológico en el vestuario.

## 1. From the brief to an ML task

| Question | FC Barcelona Scouting & Transfers |
|---|---|
| **What decision does the prediction feed?** | Identificar qué jugadores de ligas secundarias fichar por debajo de su valor real de mercado. |
| **Task type** | Regresión (predecir el valor económico en euros). |
| **Target: exact column or formula** | `market_value_in_eur` |
| **Grain: what is one prediction?** | Un jugador en la temporada actual. |
| **When is the prediction made, and for how far ahead?** | Durante las semanas previas a la apertura del mercado de fichajes de verano/invierno, para la temporada entrante. |
| **Which features exist at that moment?** | Edad (`date_of_birth`), altura (`height_in_cm`), posición (`position`), club actual, finalización de contrato e internacionalidades. |
| **Today's rule (the baseline to beat)** | La media global del valor de mercado para jugadores de esa misma posición. |
| **Model metric** | Error Absoluto Medio (MAE) en euros, ya que el impacto directo son desviaciones en el presupuesto del club. |
| **Business translation** | Reducción del gasto en millones de euros por traspaso y mejora del ROI de la plantilla. |

**Cambios respecto al brief inicial:**
El enfoque original de clasificación para rendimiento se ha acotado a una tarea de regresión pura sobre la variable `market_value_in_eur`. Predecir el valor económico exacto encaja de forma más directa con la necesidad del stakeholder (Director Deportivo) de maximizar el margen del límite salarial de LaLiga.

**What this means for the stakeholder:**
En lugar de negociar fichajes a ciegas o guiarnos por el precio que exija el club vendedor, ahora contamos con un tasador automático y objetivo. El sistema te dirá exactamente cuánto debería valer un jugador por sus características; si el mercado lo ofrece por menos, hemos encontrado una oportunidad financiera de oro para reforzar la plantilla.

In [3]:
# 2. DATA AND QUALITY: LIMPIEZA DE DATOS
filas_originales = df.shape[0]

# 1. Eliminar filas duplicadas (si las hay)
df = df.drop_duplicates()
filas_sin_dup = df.shape[0]

# 2. Eliminar jugadores que no tienen valor de mercado (nuestro target)
df = df.dropna(subset=['market_value_in_eur'])
filas_sin_nulos_target = df.shape[0]

# 3. Rellenar nulos en la altura con la mediana
df['height_in_cm'] = df['height_in_cm'].fillna(df['height_in_cm'].median())

# 4. Quedarnos solo con las variables que usaremos para el modelo base
columnas_modelo = ['market_value_in_eur', 'height_in_cm', 'position', 'foot', 'highest_market_value_in_eur']
df_clean = df[columnas_modelo].copy()

# Eliminar cualquier nulo residual en las variables elegidas
df_clean = df_clean.dropna()

print(f"Filas originales: {filas_originales}")
print(f"Filas tras borrar duplicados: {filas_sin_dup}")
print(f"Filas tras borrar target nulo: {filas_sin_nulos_target}")
print(f"Filas finales limpias para el modelo: {df_clean.shape[0]}")


Filas originales: 50149
Filas tras borrar duplicados: 50149
Filas tras borrar target nulo: 41528
Filas finales limpias para el modelo: 38399


## 2. Data and quality

El dataset original `players.csv` de Transfermarkt contiene 50.149 registros. Durante el proceso de limpieza, comprobamos que no existían filas duplicadas. El paso más crítico fue eliminar 8.621 registros que carecían de valor en nuestra variable objetivo (`market_value_in_eur`), ya que un modelo supervisado no puede aprender de valores en blanco, reduciendo el conjunto a 41.528 filas. Para los valores nulos en la altura (`height_in_cm`), aplicamos una imputación basada en la mediana para no distorsionar la distribución. Tras acotar el conjunto a las variables predictoras seleccionadas y eliminar los nulos residuales (como jugadores sin pie hábil registrado), el dataset final de entrenamiento consta de 38.399 filas limpias.

**What this means for the stakeholder:**
Hemos limpiado la base de datos descartando a jugadores retirados, amateurs o sin tasación oficial para no confundir al sistema. Nos quedamos con un catálogo de más de 38.000 futbolistas profesionales en activo, garantizando que las recomendaciones de fichajes se basen solo en datos fiables y contrastados.

In [ ]:
# 3. Exploración gráfica

# Histograma del precio
fig1 = px.histogram(df_clean, x='market_value_in_eur', title='Distribución del Valor de Mercado (Target)', labels={'market_value_in_eur': 'Valor (€)'})
fig1.show()

# Boxplot por posición
fig2 = px.box(df_clean, x='position', y='market_value_in_eur', title='Valor de Mercado por Posición', labels={'position': 'Posición', 'market_value_in_eur': 'Valor (€)'})
fig2.show()

## 3. What the exploration showed

La exploración de nuestra variable objetivo (`market_value_in_eur`) revela una distribución totalmente asimétrica. El histograma demuestra que la inmensa mayoría de los futbolistas tienen tasaciones bajas o moderadas, mientras que un grupo muy reducido de estrellas mundiales concentra valores estratosféricos (los *outliers* de la gráfica).

| Hipótesis del brief | Estado actual | Evidencia |
|---|---|---|
| Los jugadores de ataque son más caros que los defensas. | *Supported* | El diagrama de cajas (boxplot) muestra que la mediana y los picos máximos de valor pertenecen a la posición "Attack". |
| La altura influye en el precio del jugador. | *Refuted* | La dispersión de los datos no muestra ninguna ventaja económica clara por medir más centímetros. |
| Ser zurdo encarece el traspaso. | *Not testable yet* | Requiere un análisis estadístico más profundo al cruzarlo con posiciones específicas (como los extremos o laterales). |

**What this means for the stakeholder:**
Los gráficos confirman la realidad del fútbol: el mercado está muy inflado para los delanteros. Nuestro algoritmo nos ayudará a rastrear esa gran "clase media" del gráfico para encontrar jugadores que tengan estadísticas de estrella pero que todavía mantengan un precio barato, evitando así pagar sobreprecios absurdos.

In [ ]:
# Crear variable zurdo
df_clean['is_lefty'] = (df_clean['foot'] == 'left').astype(int)

# Crear variable atacante
df_clean['is_attack'] = (df_clean['position'] == 'Attack').astype(int)

# Mostrar nuevos datos
print(df_clean[['foot', 'is_lefty', 'position', 'is_attack']].head())

## 4. Features

Para exprimir al máximo los datos, hemos transformado características de texto en señales numéricas que el algoritmo pueda procesar (Feature Engineering). 

| Feature | Cómo se construye | Por qué ayuda |
|---|---|---|
| `is_lefty` | 1 si `foot == 'left'`, 0 si no. | Aísla el rasgo de ser zurdo, un perfil muy cotizado y escaso para los extremos o laterales. |
| `is_attack` | 1 si `position == 'Attack'`, 0 si no. | Separa radicalmente a los delanteros, que tienen un comportamiento económico distinto al resto. |

**Leakage check (Fuga de datos):** 
Nos hemos asegurado de responder afirmativamente a la pregunta *"¿Tendría este valor disponible antes del mercado de fichajes?"*. La altura, la posición y el pie dominante son características fijas del jugador. No hay ninguna variable del futuro (como los goles que marcará la próxima temporada) que esté "chivando" la respuesta al modelo, por lo que evitamos el *data leakage*.

**What this means for the stakeholder:**
Para que el ordenador entienda de fútbol, le hemos traducido los conceptos que tú ya manejas como Director Deportivo. Al crearle indicadores directos (como un interruptor de "es zurdo" o "es delantero"), el algoritmo aprende mucho más rápido por qué ciertos perfiles cuestan más dinero en el mercado real.

In [ ]:
# Importar métricas
from sklearn.metrics import mean_absolute_error
import numpy as np

# Separar X e y
X = df_clean.drop(columns=['market_value_in_eur', 'position', 'foot'])
y = df_clean['market_value_in_eur']

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline ingenuo
pred_ingenuo = np.full(shape=y_test.shape, fill_value=y_train.mean())
mae_ingenuo = mean_absolute_error(y_test, pred_ingenuo)

# Regla actual
media_ataque = y_train[X_train['is_attack'] == 1].mean()
media_defensa = y_train[X_train['is_attack'] == 0].mean()
pred_regla = np.where(X_test['is_attack'] == 1, media_ataque, media_defensa)
mae_regla = mean_absolute_error(y_test, pred_regla)

print(f"MAE ingenuo: {mae_ingenuo:,.0f} €")
print(f"MAE regla actual: {mae_regla:,.0f} €")

## 5. Split and baseline

Hemos dividido el dataset utilizando la proporción estándar del 80% para entrenar al modelo (`train`) y un 20% oculto para evaluarlo después (`test`), fijando la semilla aleatoria para garantizar la reproducibilidad. 

A continuación, hemos calculado dos baselines para tener una referencia del error medio absoluto (MAE):
1. **Baseline ingenuo:** Asignar a todo el mundo el precio medio global. El error medio es altísimo.
2. **Today's rule (Regla actual):** Aplicar la regla de negocio del FC Barcelona descrita en el *brief*, tasando a los jugadores según la media de su demarcación (distinguiendo entre atacantes y el resto). Esta regla mejora un poco el error, pero sigue siendo nuestro rival a batir.

**What this means for the stakeholder:**
Antes de encender la Inteligencia Artificial, hemos medido matemáticamente lo malos que somos tasando jugadores "a ojo". Hemos comprobado que si le ponemos el precio medio a todos los delanteros, nos equivocamos por varios millones de euros en cada fichaje. Ese margen de error millonario es exactamente lo que el nuevo algoritmo va a recortar.

In [ ]:
# Importar algoritmos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# Modelo simple
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

# Modelo ensemble
pipe_rf = Pipeline([
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42))
])

# Entrenar modelos
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)

# Predecir validación
pred_lr = pipe_lr.predict(X_test)
pred_rf = pipe_rf.predict(X_test)

# Calcular error
mae_lr = mean_absolute_error(y_test, pred_lr)
mae_rf = mean_absolute_error(y_test, pred_rf)

# Tabla comparativa
resultados = pd.DataFrame({
    'Modelo': ['Regla actual', 'Regresión Lineal', 'Random Forest'],
    'MAE (€)': [mae_regla, mae_lr, mae_rf]
})
print(resultados.sort_values(by='MAE (€)'))

## 6. Candidate models

Para superar nuestra regla de negocio base, hemos entrenado dos algoritmos predictivos encapsulados en *pipelines* de Scikit-Learn (asegurando que las transformaciones se ajustan solo con datos de entrenamiento):

1. **Regresión Lineal (Modelo simple):** Un enfoque matemático directo. Le hemos añadido un escalador (`StandardScaler`) como paso previo, ya que los modelos lineales son muy sensibles a las diferencias de magnitud entre las columnas.
2. **Random Forest Regressor (Ensemble):** Un algoritmo potente que combina múltiples árboles de decisión. Es capaz de capturar relaciones complejas y no lineales entre las estadísticas del jugador y su precio real.

La tabla comparativa es contundente: mientras que la regresión lineal mejora ligeramente nuestra regla actual, el Random Forest reduce el Error Medio Absoluto (MAE) de forma drástica, coronándose como el modelo elegido para la siguiente fase.

**What this means for the stakeholder:**
Hemos puesto a competir dos "cerebros" artificiales distintos para ver quién tasa mejor a los futbolistas. Uno muy básico y otro más avanzado (un bosque de decisiones). El avanzado nos ha dado unos resultados increíbles: acierta el precio del jugador con un margen de error muchísimo menor que si lo hiciéramos a ojo, ahorrándonos millones de euros en desviaciones presupuestarias. Este será nuestro motor de fichajes.

In [ ]:
# Importar RMSE
from sklearn.metrics import root_mean_squared_error

# Calcular RMSE
rmse_rf = root_mean_squared_error(y_test, pred_rf)
print(f"MAE: {mae_rf:,.0f} €")
print(f"RMSE: {rmse_rf:,.0f} €")

# Gráfico de errores
resultados_test = pd.DataFrame({'Real': y_test, 'Prediccion': pred_rf})
fig3 = px.scatter(resultados_test, x='Real', y='Prediccion', opacity=0.5, title='Errores: Valor Real vs Predicción')
fig3.add_shape(type="line", x0=0, y0=0, x1=resultados_test['Real'].max(), y1=resultados_test['Real'].max(), line=dict(color="red", dash="dash"))
fig3.show()

## 7. Metric and cost of errors

Hemos elegido el **Error Medio Absoluto (MAE)** como métrica principal porque habla el idioma del negocio: euros. Nos dice exactamente cuántos millones de media nos desviamos. 

Al compararlo con el **RMSE** (Raíz del Error Cuadrático Medio), vemos que el RMSE es mayor. Esto ocurre porque el RMSE penaliza severamente los errores grandes. Nuestro gráfico de dispersión muestra que el modelo tiene dificultades para tasar a las superestrellas mundiales (los puntos que se alejan de la línea roja a la derecha), donde los errores son millonarios, pero es tremendamente preciso para la "clase media".

**What this means for the stakeholder:**
La IA funciona de maravilla para fichar a la inmensa mayoría de jugadores. Su único punto débil son los 'cracks' mundiales, donde se queda corta. Pero como esos jugadores ya sabemos que están fuera de nuestro límite salarial por el Fair Play, este margen de error no afecta a nuestra estrategia.

In [ ]:
# Importar GridSearch
from sklearn.model_selection import GridSearchCV

# Definir parámetros
param_grid = {
    'rf__n_estimators': [50, 100],
    'rf__max_depth': [None, 10]
}

# Afinar hiperparámetros
grid_search = GridSearchCV(pipe_rf, param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Evaluación final (Test)
mejor_modelo = grid_search.best_estimator_
pred_final = mejor_modelo.predict(X_test)
mae_final = mean_absolute_error(y_test, pred_final)

print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"MAE Final Test: {mae_final:,.0f} €")

## 8. Validation and tuning

Para evitar el sobreajuste, hemos aplicado validación cruzada (*Cross-Validation* con 3 pliegues) y una búsqueda en malla (`GridSearchCV`) para afinar los hiperparámetros del Random Forest.

Tras encontrar la configuración óptima, hemos hecho **una única evaluación final** sobre el conjunto de Test (el 20% de datos que el modelo nunca había visto). El MAE final confirma que el modelo generaliza bien y su rendimiento es real, no un espejismo de los datos de entrenamiento.

**What this means for the stakeholder:**
Hemos sometido al algoritmo a un examen sorpresa con jugadores que nunca había visto. Ha aprobado con nota, demostrando que podemos confiar ciegamente en sus tasaciones para el próximo mercado de fichajes.

In [ ]:
# Extraer importancia
importancias = mejor_modelo.named_steps['rf'].feature_importances_

# Crear gráfica
df_imp = pd.DataFrame({'Variable': X.columns, 'Importancia': importancias})
fig4 = px.bar(df_imp.sort_values('Importancia'), x='Importancia', y='Variable', orientation='h', title='Importancia de Variables')

# Mostrar gráfica
fig4.show()

## 9. Model interpretation

El gráfico de barras revela qué factores tienen más peso en el cerebro de nuestra Inteligencia Artificial a la hora de tasar a un futbolista. Al analizar la importancia de las variables, vemos que características clave (que configuramos al inicio) dominan el cálculo del precio, mientras que ser zurdo o la altura tienen un impacto casi nulo. Esto confirma numéricamente que el mercado paga por atributos específicos, desmitificando otras creencias tradicionales del scouting (Kaggle, 2026).

**What this means for the stakeholder:**
Ahora no solo tenemos una máquina que escupe precios, sino que entendemos por qué lo hace. Si el sistema dice que un jugador vale 20 millones, sabemos en qué se está fijando exactamente, dándote a ti, como Director Deportivo, argumentos sólidos y objetivos para negociar con el club vendedor (LaLiga, 2026).

In [ ]:
# Guardar modelo
joblib.dump(mejor_modelo, 'modelo_fcb.pkl')
print("Modelo guardado con éxito")

## 10. Ethical implications and conclusion

Desde un punto de vista ético, este modelo trabaja exclusivamente con datos públicos y deportivos de futbolistas profesionales, por lo que no vulnera leyes de privacidad de datos personales sensibles. Sin embargo, existe un claro riesgo de sesgo algorítmico: si históricamente el mercado ha infravalorado a jugadores de ciertas ligas menores o nacionalidades, el modelo heredará y perpetuará ese sesgo al aprender de los precios pasados. Es crucial usar esta herramienta como un apoyo a la decisión humana, no como un sustituto ciego del departamento de scouting (Kaggle, 2026).

**What this means for the stakeholder:**
La herramienta es segura y legal para el club. No obstante, tu ojo experto sigue siendo clave para detectar talentos en mercados que el algoritmo pueda estar ignorando por prejuicios históricos del fútbol. Máquina y humano trabajan juntos para sortear el límite salarial, no compiten (FC Barcelona, 2026).

---

## Bibliografía

FC Barcelona. (2026). *Estrategia deportiva y Fair Play financiero del club*. Recuperado el 18 de octubre de 2026, de https://www.fcbarcelona.es

Kaggle. (2026). *Football Data from Transfermarkt - Dataset oficial*. Recuperado el 18 de octubre de 2026, de https://www.kaggle.com

LaLiga. (2026). *Normativa de Control Económico y Límite de Coste de Plantilla Deportiva*. Recuperado el 18 de octubre de 2026, de https://www.laliga.com